# PPO Training Example on ImprovedUltrastickEnv

This notebook demonstrates training a PPO agent from `tensoraerospace/agent/ppo/model.py` on the `ImprovedUltrastickEnv` environment from `tensoraerospace/envs/ultrastick.py`.

**Environment features:**
- Aircraft control via elevator deflection
- Pitch angle theta tracking with a sinusoidal reference signal
- Normalized action and observation spaces [-1, 1]
- LQR-like reward function

**Algorithm features:**
- Continuous actions with Gaussian policy
- Clipped surrogate objective for stable updates
- tqdm progress bar and TensorBoard logging
- Checkpointing for resuming training

In [ ]:
# Optional installs (run if needed)
# %pip install tensorboard tqdm --quiet

import os
import numpy as np
import torch

from tensoraerospace.envs.ultrastick import ImprovedUltrastickEnv
from tensoraerospace.agent.ppo.model import PPO

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Using device:", DEVICE)

# Reproducibility
np.random.seed(42)
_ = torch.manual_seed(42)


In [ ]:
# Build environment
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

# Time base
from tensoraerospace.signals.standart import sinusoid_vertical_shift
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

# Time base
dt = 0.1
_tp = generate_time_period(tn=20, dt=dt)
tps = convert_tp_to_sec_tp(_tp, dt=dt)
number_time_steps = len(_tp)

# Reference theta (radians)
reference_signals = np.reshape(
    sinusoid_vertical_shift(
        tp=np.asarray(tps),
        frequency=0.05,
        amplitude=np.deg2rad(1.0),
        vertical_shift=0.0,
    ),
    [1, -1],
)

# Initial state [rho (km), rho_dot (m/s), theta_dot (rad/s)]
# Start at Earth radius with target angular velocity
init_state = np.array([0.0,0.0, 0.0, 0.0, 0.0], dtype=np.float32)

# Env
env = ImprovedUltrastickEnv(
    initial_state=init_state,
    reference_signal=reference_signals,
    number_time_steps=number_time_steps,
    dt=dt,
    initial_elevator_deg= 0.0
)

obs, info = env.reset()
print("Obs shape:", np.array(obs).shape)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)
print("\nEnvironment configuration:")
print(f"  dt: {env.dt} sec")
print(f"  Steps: {number_time_steps}")
print(f"  Duration: {number_time_steps * env.dt:.1f} sec")
print(f"  Reward scale: {env.reward_scale}")
print(f"  Survival bonus: +0.1 per step")


In [ ]:
# Create PPO agent (tuned hyperparams)
# Hyperparameters for ComSat:
# - Higher gamma (0.99) for long-term orbital stability
# - Larger rollout_len (2048) for diverse trajectory collection
# - Moderate clip_param (0.2) balances exploration and stability
# - Lower entropy_coef (0.01) reduces randomness for precise control
# - Smaller learning rates for satellite control
# - normalize_obs=True for stable learning with normalized observations

agent = PPO(
    env=env,
    gamma=0.995,
    max_episodes=1000,
    rollout_len=4096,
    clip_pram=0.2,
    num_epochs=4,
    batch_size=256,
    entropy_coef=0.02,
    actor_lr=1e-4,
    critic_lr=3e-4,
    gae_lambda=0.95,
    max_grad_norm=0.5,
    target_kl=0.03,
    normalize_obs=True,
    normalize_reward=True,
    seed=42)

print("PPO agent created")
# print(f"Actor network: {agent.policy_network}")
# print(f"\nValue network: {agent.value_network}")


## Training

**Training recommendations:**

1. **Start simple** - use a constant target angular velocity
2. **Monitor metrics:**
   - `mean_reward` - should increase and stabilize
   - `entropy` - should decrease as the policy converges
   - `policy_loss` and `value_loss` - should decrease
   - `approx_kl` - should remain small (~0.01)

3. **Signs of successful training:**
   - Reward > -10 (agent avoids termination)
   - Mean reward grows to ~-5 or higher
   - Agent completes all 200 steps without termination

4. **If training stalls:**
   - Check metrics in TensorBoard: `tensorboard --logdir runs/`
   - Halve `actor_lr` and `critic_lr`
   - Increase `rollout_len` to 4096
   - Try `normalize_reward=True`

In [ ]:
# Train the agent
# This will take some time depending on your hardware
# Progress will be shown via tqdm progress bar
# Metrics are logged to TensorBoard in runs/ directory

print("Starting training...")
print(f"Total episodes: {agent.max_episodes}")
print(f"Rollout length: {agent.rollout_len}")
print(f"Expected total steps: ~{agent.max_episodes * agent.rollout_len // number_time_steps}")
print("\nMonitor training: tensorboard --logdir runs/\n")

# Train
agent.train()

print("\nTraining completed!")


In [ ]:
# Evaluation run
obs, info = env.reset()
done = False
ret = 0.0
step = 0

while not done and step < number_time_steps:
    action = agent.act(obs, deterministic=False)
    step_return = env.step(action)
    if len(step_return) == 5:
        obs, reward, terminated, truncated, info = step_return
        done = terminated or truncated
    else:
        obs, reward, terminated, info = step_return[:4]
        done = terminated
    ret += float(reward)
    step += 1

print(f"Evaluation reward: {ret:.3f}")
step

In [ ]:
step

In [ ]:
# Plot theta (pitch angle) tracking
env.unwrapped.model.plot_transient_process('theta', tps, reference_signals[0], to_deg=True, figsize=(15,4))

In [ ]:
# Plot elevator control
env.unwrapped.model.plot_control(control_name='ele', time=tps, to_deg=True, figsize=(15,4))


## Evaluation of the Trained Model

Test the trained agent on a single episode and collect data for visualization.